<a href="https://colab.research.google.com/github/enilt/langgraph-alura/blob/main/Aula_01_ReAct_Groq_(v1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente ReAct com Groq — Function Calling Nativo

**Adaptado do curso Alura "Orquestração de agentes e multiagentes com LangGraph" (Aula 01 – ReAct).**

A versão original usava o Google Gemini. Aqui trocamos por **Groq** e, em vez de escrever o
laço ReAct "na mão" por texto, usamos o recurso de **chamada de ferramentas nativa**
(*function calling*). Vantagens:

- Muito mais confiável com os modelos de raciocínio (GPT-OSS) que hoje são o padrão na Groq.
- O modelo **não inventa dados**: quem devolve o resultado real é a ferramenta.
- É o mesmo mecanismo que o **N8N** usa no nó "AI Agent", então dá para reaproveitar depois.

> Funciona tanto no **Google Colab** quanto no **VS Code** (com um arquivo `.env`).

## 1. Instalação das dependências

Só precisamos do cliente oficial da Groq. O `python-dotenv` ajuda a ler a chave no VS Code.

In [ ]:
# No Colab roda direto. No VS Code, pode rodar uma vez no terminal do ambiente.
%pip install -q groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.8 MB/s eta 0:00:00


## 2. Configurar a chave da API da Groq

Crie uma chave gratuita em <https://console.groq.com/keys>.

A função abaixo procura a chave em **três lugares**, nesta ordem, então o mesmo notebook
funciona em qualquer ambiente sem alterar código:

1. **Variável de ambiente** `GROQ_API_KEY` (ideal para VS Code / N8N).
2. **Secrets do Google Colab** (ícone da chavinha 🔑 na barra lateral → nome `GROQ_API_KEY`).
3. **Arquivo `.env`** na mesma pasta do notebook (VS Code local).

In [ ]:
import os

def get_secret(nome: str) -> str:
    """Busca um segredo em: variavel de ambiente -> Colab Secrets -> arquivo .env."""
    # 1) Variavel de ambiente
    valor = os.environ.get(nome)
    if valor:
        return valor

    # 2) Google Colab (userdata)
    try:
        from google.colab import userdata  # type: ignore
        valor = userdata.get(nome)
        if valor:
            return valor
    except Exception:
        pass

    # 3) Arquivo .env (VS Code local)
    try:
        from dotenv import load_dotenv
        load_dotenv()
        valor = os.environ.get(nome)
        if valor:
            return valor
    except Exception:
        pass

    raise RuntimeError(
        f"Nao encontrei o segredo '{nome}'. Configure de uma destas formas:\n"
        f"  - Colab: icone da chave na lateral, nome {nome}\n"
        f"  - VS Code: crie um arquivo .env com {nome}=sua_chave\n"
        f"  - Terminal: export {nome}=sua_chave"
    )

GROQ_API_KEY = get_secret("GROQ_API_KEY")
print("Chave carregada com sucesso! (comeca com:", GROQ_API_KEY[:6] + "...)")

Chave carregada com sucesso! (comeca com: gsk_AO...)


## 3. Criar o cliente da Groq e escolher o modelo

Atenção: a Groq **desativou** os modelos `llama-3.3-70b-versatile` e `llama-3.1-8b-instant`
no plano gratuito em agosto/2026. Os substitutos recomendados são os **GPT-OSS**, que são
modelos de raciocínio e funcionam muito bem com function calling.

In [ ]:
from groq import Groq

cliente = Groq(api_key=GROQ_API_KEY)

# Modelo padrao (recomendado pela Groq). Alternativas gratuitas:
#   openai/gpt-oss-20b   -> mais rapido / mais barato
#   qwen/qwen3-32b       -> alternativa
MODELO_GROQ = "openai/gpt-oss-120b"

print("Cliente Groq pronto. Modelo:", MODELO_GROQ)

Cliente Groq pronto. Modelo: openai/gpt-oss-120b


## 4. As ferramentas (o que o agente sabe fazer)

Aqui criamos algumas ferramentas de exemplo, genéricas, para mostrar o padrão ReAct
funcionando. Depois é só trocar pelo que você precisar (consulta a um banco, uma API, etc.).

**Regra de ouro:** o modelo **nunca** inventa resultados. Ele só chama estas funções,
e são elas que devolvem os dados reais.

In [ ]:
import json
import math
from datetime import datetime

def calcular(expressao: str) -> str:
    """Calcula uma expressao matematica simples de forma segura."""
    # Permite apenas nomes matematicos (sqrt, sin, pi, ...), sem acesso ao sistema
    permitido = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    permitido["abs"] = abs
    permitido["round"] = round
    try:
        resultado = eval(expressao, {"__builtins__": {}}, permitido)
        return json.dumps({"expressao": expressao, "resultado": resultado}, ensure_ascii=False)
    except Exception as e:
        return json.dumps({"erro": f"Nao consegui calcular: {e}"}, ensure_ascii=False)

def consultar_clima(cidade: str) -> str:
    """Retorna o clima (SIMULADO) de uma cidade. Troque por uma API real depois."""
    base = {
        "sao paulo": {"temp_c": 24, "condicao": "parcialmente nublado"},
        "rio de janeiro": {"temp_c": 31, "condicao": "ensolarado"},
        "curitiba": {"temp_c": 18, "condicao": "chuvoso"},
    }
    dados = base.get(cidade.strip().lower())
    if not dados:
        return json.dumps({"erro": f"Sem dados de clima para '{cidade}'."}, ensure_ascii=False)
    return json.dumps({"cidade": cidade, **dados}, ensure_ascii=False)

def converter_temperatura(valor: float, de: str, para: str) -> str:
    """Converte temperatura entre C, F e K."""
    de, para = de.strip().upper(), para.strip().upper()
    # normaliza tudo para Celsius primeiro
    if de == "C":   c = float(valor)
    elif de == "F": c = (float(valor) - 32) * 5 / 9
    elif de == "K": c = float(valor) - 273.15
    else: return json.dumps({"erro": f"Unidade de origem invalida: {de}"}, ensure_ascii=False)
    if para == "C":   out = c
    elif para == "F": out = c * 9 / 5 + 32
    elif para == "K": out = c + 273.15
    else: return json.dumps({"erro": f"Unidade de destino invalida: {para}"}, ensure_ascii=False)
    return json.dumps({"valor": valor, "de": de, "para": para, "resultado": round(out, 2)}, ensure_ascii=False)

def data_hora_atual() -> str:
    """Retorna a data e hora atuais do sistema."""
    agora = datetime.now()
    return json.dumps({"data_hora": agora.strftime("%d/%m/%Y %H:%M:%S")}, ensure_ascii=False)

### 4.1 Descrever as ferramentas para o modelo

O function calling nativo precisa de uma "ficha técnica" (schema) de cada ferramenta:
nome, para que serve e quais parâmetros recebe. É assim que o modelo sabe **quando** e
**como** chamar cada função. Esse formato é o mesmo padrão usado pelo N8N.

In [ ]:
FERRAMENTAS = [
    {
        "type": "function",
        "function": {
            "name": "calcular",
            "description": "Calcula uma expressao matematica (ex: '2*(3+4)', 'sqrt(16)').",
            "parameters": {
                "type": "object",
                "properties": {
                    "expressao": {"type": "string", "description": "A expressao a calcular"}
                },
                "required": ["expressao"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_clima",
            "description": "Consulta o clima atual de uma cidade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "cidade": {"type": "string", "description": "Nome da cidade"}
                },
                "required": ["cidade"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "converter_temperatura",
            "description": "Converte uma temperatura entre C (Celsius), F (Fahrenheit) e K (Kelvin).",
            "parameters": {
                "type": "object",
                "properties": {
                    "valor": {"type": "number", "description": "Valor da temperatura"},
                    "de":    {"type": "string", "description": "Unidade de origem: C, F ou K"},
                    "para":  {"type": "string", "description": "Unidade de destino: C, F ou K"},
                },
                "required": ["valor", "de", "para"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "data_hora_atual",
            "description": "Retorna a data e a hora atuais.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

# Mapa nome -> funcao Python, usado pelo agente para executar a chamada
FUNCOES_DISPONIVEIS = {
    "calcular": calcular,
    "consultar_clima": consultar_clima,
    "converter_temperatura": converter_temperatura,
    "data_hora_atual": data_hora_atual,
}
print("Ferramentas registradas:", list(FUNCOES_DISPONIVEIS.keys()))

Ferramentas registradas: ['calcular', 'consultar_clima', 'converter_temperatura', 'data_hora_atual']


## 5. O agente (laço de function calling)

O fluxo é este, e ele é o mesmo "ReAct" do curso, só que automático:

1. Mandamos a pergunta + a lista de ferramentas para o modelo.
2. Se o modelo decide **usar uma ferramenta**, ele responde com uma "chamada de função".
3. Nós **executamos** a função Python de verdade e devolvemos o resultado.
4. Repetimos até o modelo ter dados suficientes e escrever a **resposta final**.

O `SYSTEM_PROMPT` dá as regras de conduta — principalmente: **não inventar dados**.

In [ ]:
SYSTEM_PROMPT = (
    "Voce e um assistente util. "
    "Use SEMPRE as ferramentas disponiveis para obter dados (calculos, clima, data/hora, etc.). "
    "NUNCA invente resultados: se precisar de um dado, chame a ferramenta apropriada. "
    "Responda em portugues do Brasil, de forma clara e objetiva."
)

def executar_agente(pergunta: str, verbose: bool = True, max_passos: int = 8) -> str:
    """Roda o agente com function calling nativo da Groq ate obter a resposta final."""
    mensagens = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pergunta},
    ]

    for passo in range(1, max_passos + 1):
        resposta = cliente.chat.completions.create(
            model=MODELO_GROQ,
            messages=mensagens,
            tools=FERRAMENTAS,
            tool_choice="auto",
            temperature=0,               # deterministico
            reasoning_effort="low",      # modelos GPT-OSS sao de raciocinio
        )
        msg = resposta.choices[0].message

        # Se o modelo NAO pediu ferramenta, e a resposta final
        if not msg.tool_calls:
            if verbose:
                print(f"\n=== Resposta final (passo {passo}) ===")
            return msg.content or "(sem conteudo)"

        # Guardamos a fala do assistente (com os pedidos de ferramenta)
        mensagens.append(msg)

        # Executamos cada ferramenta pedida
        for chamada in msg.tool_calls:
            nome = chamada.function.name
            try:
                args = json.loads(chamada.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose:
                print(f"[passo {passo}] Ferramenta: {nome}({args})")

            funcao = FUNCOES_DISPONIVEIS.get(nome)
            resultado = funcao(**args) if funcao else json.dumps({"erro": f"Ferramenta {nome} nao existe."})

            if verbose:
                print(f"          -> {resultado}")

            mensagens.append({
                "role": "tool",
                "tool_call_id": chamada.id,
                "name": nome,
                "content": resultado,
            })

    return "Atingi o limite de passos sem concluir. Tente reformular a pergunta."

## 6. Testando o agente

Vamos ver o agente trabalhando passo a passo (`verbose=True`). Repare que cada resultado
vem de uma **ferramenta**, nunca da imaginação do modelo.

In [ ]:
print(executar_agente("Quanto e 15% de 340 mais 12?"))

[passo 1] Ferramenta: calcular({'expressao': '(0.15*340)+12'})
          -> {"expressao": "(0.15*340)+12", "resultado": 63.0}

=== Resposta final (passo 2) ===
15% de 340 é 51. Somando 12, o resultado é **63**.


In [ ]:
print(executar_agente("Como esta o clima em Curitiba? E quanto isso da em Fahrenheit?"))

[passo 1] Ferramenta: consultar_clima({'cidade': 'Curitiba'})
          -> {"cidade": "Curitiba", "temp_c": 18, "condicao": "chuvoso"}
[passo 2] Ferramenta: converter_temperatura({'de': 'C', 'para': 'F', 'valor': 18})
          -> {"valor": 18, "de": "C", "para": "F", "resultado": 64.4}

=== Resposta final (passo 3) ===
O clima atual em Curitiba está **chuvoso** com temperatura de **18 °C**.  

Convertendo para Fahrenheit, a temperatura corresponde a **64,4 °F**.


In [ ]:
print(executar_agente("Que dia e hoje?"))

[passo 1] Ferramenta: data_hora_atual({})
          -> {"data_hora": "23/09/2026 17:54:26"}

=== Resposta final (passo 2) ===
Hoje é 23 de setembro de 2026.


## 7. Modo "saída limpa"

Quando você já confia no agente e quer só o resultado (sem ver os passos internos),
use `verbose=False`.

In [ ]:
def responder(pergunta: str) -> str:
    """Atalho: roda o agente sem mostrar os passos internos."""
    return executar_agente(pergunta, verbose=False)

print(responder("Converta 100 F para Celsius."))

100 °F correspondem a aproximadamente **37,78 °C**.


## 8. Próximos passos (adaptar e usar no N8N)

- **Trocar as ferramentas de exemplo pelas suas:** basta alterar o *corpo* das funções
  (ex.: consultar um banco, chamar uma API). A "ficha técnica" (`FERRAMENTAS`) segue o
  mesmo formato.
- **N8N:** este mesmo desenho (ferramentas com nome + descrição + parâmetros) é exatamente
  o que o nó **AI Agent** espera. Você recria cada função como um nó/"Tool" e conecta ao
  agente, apontando para o modelo da Groq.
- **Confiabilidade:** como o modelo nunca inventa resultados, a fonte da verdade são
  sempre suas funções.